# Calcul des embeddings (pas-à-pas)

Ce notebook reprend le script `src/compute_embeddings.py` pour exécuter le calcul des embeddings étape par étape.

## 1) Imports et accès au projet

In [1]:
# Imports standard + TensorFlow/NumPy.
from pathlib import Path
import sys
import numpy as np
import tensorflow as tf

# Détecte la racine du projet (dossier qui contient src).
project_root = Path.cwd()
if (project_root / 'src').exists():
    pass
elif (project_root.parent / 'src').exists():
    project_root = project_root.parent
else:
    raise RuntimeError('Impossible de trouver le dossier src depuis le notebook.')

# Ajoute la racine projet au PYTHONPATH pour importer src/.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print('Project root:', project_root)


Project root: /Users/lewagon/code/Lionel-Nokia/art-xplain/art-xplain


## 2) Charger config et utilitaires

In [2]:
# Utilitaires projet + fonctions du script compute_embeddings.
from src.utils import load_config, ensure_dir
from src.compute_embeddings import collect_records, build_dataset

# Charge la config YAML.
cfg = load_config(project_root / 'config.yaml')
keras_root = project_root / cfg['paths']['keras_root']
embeddings_root = project_root / cfg['paths']['embeddings_root']
models_root = project_root / cfg['paths']['models_root']

print('keras_root:', keras_root)
print('embeddings_root:', embeddings_root)
print('models_root:', models_root)


keras_root: /Users/lewagon/code/Lionel-Nokia/art-xplain/art-xplain/data/out
embeddings_root: /Users/lewagon/code/Lionel-Nokia/art-xplain/art-xplain/embeddings
models_root: /Users/lewagon/code/Lionel-Nokia/art-xplain/art-xplain/models


## 3) Collecter les chemins d'images et labels

In [3]:
# Collecte des chemins d'images + labels depuis keras_root.
file_paths, labels, class_names = collect_records(keras_root)

print('Total images:', len(file_paths))
print('Nb classes:', len(class_names))
print('Exemple classe:', class_names[0])
file_paths[:3]


Total images: 60598
Nb classes: 10
Exemple classe: Abstract_Expressionism


array(['/Users/lewagon/code/Lionel-Nokia/art-xplain/art-xplain/data/out/train/Abstract_Expressionism/aaron-siskind_acolman-1-1955.jpg',
       '/Users/lewagon/code/Lionel-Nokia/art-xplain/art-xplain/data/out/train/Abstract_Expressionism/aaron-siskind_feet-102-1957.jpg',
       '/Users/lewagon/code/Lionel-Nokia/art-xplain/art-xplain/data/out/train/Abstract_Expressionism/aaron-siskind_gloucester-16a-1944.jpg'],
      dtype=object)

## 4) Construire le dataset TensorFlow

In [4]:
# Paramètres image/batch pour le dataset tf.data.
img_size = int(cfg['model']['img_size'])
batch_size = int(cfg.get('train', {}).get('batch_size', 32))

# Construit un dataset tf.data (images + labels).
ds = build_dataset(file_paths, labels, img_size=img_size, batch_size=batch_size)
ds


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

## 5) Charger l'encodeur

In [5]:
# Charge le modèle entraîné (safe_mode=False pour UnitNormalization/Lambda).
encoder = tf.keras.models.load_model(models_root / 'encoder.keras', compile=False, safe_mode=False)
encoder.summary()


Model: "styledna_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 7, 7, 1280)     │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ l2norm (UnitNormalization)      │ (None, 256)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,247,248 (23.83 MB)

 Trainable params: 2,630,848 (10.04 MB)

 Non-trainable params: 3,616,400 (13.80 MB)

## 6) Calculer les embeddings

In [6]:
# Boucle sur les batches pour générer les embeddings.
all_embs = []
all_labels = []

for batch_imgs, batch_labels in ds:
    embs = encoder(batch_imgs, training=False).numpy()
    all_embs.append(embs)
    all_labels.append(batch_labels.numpy())

# Assemble tous les batches en un seul tableau.
embeddings = np.vstack(all_embs)
labels_out = np.concatenate(all_labels)

print('embeddings shape:', embeddings.shape)
print('labels shape:', labels_out.shape)


embeddings shape: (60598, 256)
labels shape: (60598,)


2026-03-12 17:58:38.677771: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 7) Sauvegarder les fichiers numpy

In [7]:
# Crée le dossier embeddings et sauvegarde les fichiers .npy.
ensure_dir(embeddings_root)
np.save(embeddings_root / 'vectors.npy', embeddings)
np.save(embeddings_root / 'labels.npy', labels_out)
np.save(embeddings_root / 'filenames.npy', file_paths, allow_pickle=True)
np.save(embeddings_root / 'classnames.npy', class_names, allow_pickle=True)

print('Saved embeddings to:', embeddings_root.resolve())


Saved embeddings to: /Users/lewagon/code/Lionel-Nokia/art-xplain/art-xplain/embeddings
